# Measurement model for quadrotor with quaternion state

This notebook defines the measurement model

$$
y = h(x)
$$

and computes the Jacobian

$$
C = \frac{\partial h}{\partial x}
$$

for a quadrotor whose state uses a quaternion with **scalar-first** convention:

$$
q = \begin{bmatrix} q_w & q_x & q_y & q_z \end{bmatrix}^T
$$

The state is

$$
x =
\begin{bmatrix}
p_x & p_y & p_z &
v_x & v_y & v_z &
q_w & q_x & q_y & q_z &
\omega_x & \omega_y & \omega_z
\end{bmatrix}^T
$$

The available sensors are:
- position sensor $(p_x, p_y, p_z)$ in **world frame**
- gyroscope $(\omega_x, \omega_y, \omega_z)$ in **body frame**
- accelerometer $(a_x, a_y, a_z)$ in **body frame**


In [2]:
import sympy as sp
from IPython.display import display, Math

sp.init_printing(use_unicode=True)


## 1. Symbolic state and parameters


In [3]:
# Position in world frame
px, py, pz = sp.symbols('p_x p_y p_z', real=True)

# Linear velocity in world frame
vx, vy, vz = sp.symbols('v_x v_y v_z', real=True)

# Quaternion q = [qw, qx, qy, qz] (scalar first)
qw, qx, qy, qz = sp.symbols('q_w q_x q_y q_z', real=True)

# Body rates
wx, wy, wz = sp.symbols('omega_x omega_y omega_z', real=True)

# Input
T = sp.symbols('T', real=True)

# Parameters
m, g = sp.symbols('m g', positive=True, real=True)

# State vector
X = sp.Matrix([
    px, py, pz,
    vx, vy, vz,
    qw, qx, qy, qz,
    wx, wy, wz
])

display(Math(r'x = ' + sp.latex(X)))


<IPython.core.display.Math object>

## 2. Rotation matrix from quaternion

Using the quaternion convention

$$
q = \begin{bmatrix} q_w & q_x & q_y & q_z \end{bmatrix}^T
$$

the rotation matrix from **body frame to world frame** is
$$
R_{bw}(q).
$$


In [4]:
R_bw = sp.Matrix([
    [1 - 2*(qy**2 + qz**2),   2*(qx*qy - qw*qz),     2*(qx*qz + qw*qy)],
    [2*(qx*qy + qw*qz),       1 - 2*(qx**2 + qz**2), 2*(qy*qz - qw*qx)],
    [2*(qx*qz - qw*qy),       2*(qy*qz + qw*qx),     1 - 2*(qx**2 + qy**2)]
])

R_wb = R_bw.T

display(Math(r'R_{bw}(q) = ' + sp.latex(R_bw)))


<IPython.core.display.Math object>

## 3. Accelerometer measurement model

The thrust acts along the body $z$-axis:

$$
F_T^b = \begin{bmatrix} 0 \\ 0 \\ T \end{bmatrix}
$$

Gravity in world frame:

$$
g^w = \begin{bmatrix} 0 \\ 0 \\ -g \end{bmatrix}
$$

World-frame translational acceleration:

$$
a^w = g^w + \frac{1}{m} R_{bw} F_T^b
$$

The accelerometer measures the **body-frame acceleration** (the world-frame
acceleration rotated into the body frame, including the gravity term):

$$
a_{\mathrm{meas}}^b = R_{wb}\, a^w
= R_{wb} g^w + \frac{1}{m} F_T^b
$$


In [5]:
g_w = sp.Matrix([0, 0, -g])
F_thrust_b = sp.Matrix([0, 0, T])

a_w = g_w + (1 / m) * (R_bw * F_thrust_b)
a_b_meas = sp.simplify(R_wb * a_w)

display(Math(r'a^w = ' + sp.latex(a_w)))
display(Math(r'a_{\mathrm{meas}}^b = ' + sp.latex(a_b_meas)))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Measurement model $y = h(x)$

The measurement vector is

$$
y =
\begin{bmatrix}
p_x \\ p_y \\ p_z \\
\omega_x \\ \omega_y \\ \omega_z \\
a_x \\ a_y \\ a_z
\end{bmatrix}
= h(x)
$$


In [6]:
h = sp.Matrix([
    px,
    py,
    pz,
    wx,
    wy,
    wz,
    a_b_meas[0],
    a_b_meas[1],
    a_b_meas[2]
])

display(Math(r'h(x) = ' + sp.latex(h)))


<IPython.core.display.Math object>

## 5. Jacobian $C = \partial h / \partial x$

For the linearized measurement model,

$$
y \approx h(\bar{x}) + C (x - \bar{x}),
\qquad
C = \frac{\partial h}{\partial x}
$$


In [7]:
C = sp.simplify(h.jacobian(X))
display(Math(r'C = \frac{\partial h}{\partial x} = ' + sp.latex(C)))


<IPython.core.display.Math object>

## 6. Quaternion normalization constraint

The quaternion must satisfy

$$
q_w^2 + q_x^2 + q_y^2 + q_z^2 = 1
$$


In [8]:
# Quaternion partial derivative normalization factor
quat_vec = sp.Matrix([qw, qx, qy, qz])
q_norm = quat_vec.norm()
partial_q_expansion = (sp.Matrix.ones(4, 4) - quat_vec * quat_vec.transpose() / q_norm**2) / q_norm
partial_q_expansion


⎡               2                                                              ↪
⎢            q_w                            q_w⋅qₓ                         q_w ↪
⎢- ──────────────────────── + 1  - ──────────────────────── + 1  - ─────────── ↪
⎢     2     2      2      2           2     2      2      2           2     2  ↪
⎢  q_w  + qₓ  + q_y  + q_z         q_w  + qₓ  + q_y  + q_z         q_w  + qₓ   ↪
⎢──────────────────────────────  ──────────────────────────────  ───────────── ↪
⎢   __________________________      __________________________      __________ ↪
⎢  ╱    2     2      2      2      ╱    2     2      2      2      ╱    2      ↪
⎢╲╱  q_w  + qₓ  + q_y  + q_z     ╲╱  q_w  + qₓ  + q_y  + q_z     ╲╱  q_w  + qₓ ↪
⎢                                                                              ↪
⎢                                              2                               ↪
⎢           q_w⋅qₓ                           qₓ                             qₓ ↪
⎢- ──────────────────────── 

## 7. Optional pretty-printed symbolic expressions


In [9]:
print('State vector X =')
sp.pprint(X)

print('\nRotation matrix R_bw =')
sp.pprint(R_bw)

print('\nMeasurement model h(x) =')
sp.pprint(h)

print('\nJacobian C = dh/dx =')
sp.pprint(C)


State vector X =
⎡pₓ ⎤
⎢   ⎥
⎢p_y⎥
⎢   ⎥
⎢p_z⎥
⎢   ⎥
⎢vₓ ⎥
⎢   ⎥
⎢v_y⎥
⎢   ⎥
⎢v_z⎥
⎢   ⎥
⎢q_w⎥
⎢   ⎥
⎢qₓ ⎥
⎢   ⎥
⎢q_y⎥
⎢   ⎥
⎢q_z⎥
⎢   ⎥
⎢ωₓ ⎥
⎢   ⎥
⎢ω_y⎥
⎢   ⎥
⎣ω_z⎦

Rotation matrix R_bw =
⎡       2        2                                                  ⎤
⎢- 2⋅q_y  - 2⋅q_z  + 1  -2⋅q_w⋅q_z + 2⋅qₓ⋅q_y  2⋅q_w⋅q_y + 2⋅qₓ⋅q_z ⎥
⎢                                                                   ⎥
⎢                             2        2                            ⎥
⎢2⋅q_w⋅q_z + 2⋅qₓ⋅q_y   - 2⋅qₓ  - 2⋅q_z  + 1   -2⋅q_w⋅qₓ + 2⋅q_y⋅q_z⎥
⎢                                                                   ⎥
⎢                                                    2        2     ⎥
⎣-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z  2⋅q_w⋅qₓ + 2⋅q_y⋅q_z   - 2⋅qₓ  - 2⋅q_y  + 1 ⎦

Measurement model h(x) =
⎡                                   pₓ                                    ⎤
⎢                                                                         ⎥
⎢                                   p_y                                